# D9-F0 B/E Long24 Full Kaggle Run

Muc tieu:
- Chay D9-F0-B va D9-F0-E long24 tren Kaggle GPU.
- Dung graph_repo input co san.
- Khong rebuild graph_repo.
- Khong chay Stage 2 classifier.
- Khong dung batch cap trong full run.
- Kiem tra resolved config de tranh dinh smoke override.

In [ ]:
import os, sys, subprocess, json, textwrap, shutil, time
from pathlib import Path

print("Python:", sys.version)
try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch import error:", e)

print("CWD:", os.getcwd())
print("Kaggle input:")
input_root = Path("/kaggle/input")
print(list(input_root.glob("*")) if input_root.exists() else "Missing /kaggle/input")
print("/kaggle/working:")
print(list(Path("/kaggle/working").glob("*")) if Path("/kaggle/working").exists() else "Missing /kaggle/working")

In [ ]:
# Sua GRAPH_REPO_PATH theo Kaggle Input thuc te cua ban.
# Notebook khong hard-code dataset name vi Kaggle Input co the khac nhau moi lan attach.
GRAPH_REPO_PATH = Path("/kaggle/input/YOUR_GRAPH_REPO_DATASET/graph_repo")
REPO_ROOT = Path("/kaggle/working/sgu-2026-facial-expression-recognition")
OUTPUT_ROOT = Path("/kaggle/working/outputs_d9_f0_be_long24_full")
FORCE_OVERWRITE = False

assert GRAPH_REPO_PATH.exists(), f"Missing GRAPH_REPO_PATH: {GRAPH_REPO_PATH}"
print("GRAPH_REPO_PATH:", GRAPH_REPO_PATH)
manifest_path = GRAPH_REPO_PATH / "manifest.pt"
shared_path = GRAPH_REPO_PATH / "shared" / "shared_graph.pt"
assert manifest_path.exists(), f"Missing graph_repo manifest: {manifest_path}"
assert shared_path.exists(), f"Missing shared graph: {shared_path}"
for split in ("train", "val", "test"):
    split_dir = GRAPH_REPO_PATH / split
    assert split_dir.exists(), f"Missing split directory: {split_dir}"
    chunks = sorted(split_dir.glob("chunk_*.pt"))
    assert chunks, f"Missing graph chunks for split={split}: {split_dir}/chunk_*.pt"
    print(f"split {split}: {len(chunks)} chunks, first={chunks[0].name}")

In [ ]:
# Sua REPO_ROOT neu repo nam o path khac trong /kaggle/working.
if not REPO_ROOT.exists():
    candidates = [p for p in Path("/kaggle/working").glob("*") if p.is_dir() and (p / "scripts" / "train_motif_discovery_stage1.py").exists()]
    print("Repo candidates:", candidates)
    if candidates:
        REPO_ROOT = candidates[0]
assert (REPO_ROOT / "scripts" / "train_motif_discovery_stage1.py").exists(), f"Missing repo at REPO_ROOT: {REPO_ROOT}"
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)
print("CWD:", os.getcwd())
if (REPO_ROOT / ".git").exists():
    subprocess.run(["git", "status", "--short"], check=False)
else:
    print("No .git directory; skip git status")
print("Repo root listing:")
print([p.name for p in REPO_ROOT.iterdir()][:80])

In [ ]:
import yaml

CONFIGS = {
    "B": {
        "name": "d9_f0_b_intensity_xy_full_edge_kaggle_long24_full",
        "path": Path("configs/experiments/d9_f0_b_intensity_xy_full_edge_kaggle_long24_full.yaml"),
        "node_indices": [0, 1, 2],
        "edge_indices": [0, 1, 2, 3, 4],
        "node_dim": 3,
        "edge_dim": 5,
    },
    "E": {
        "name": "d9_f0_e_no_xy_full_edge_kaggle_long24_full",
        "path": Path("configs/experiments/d9_f0_e_no_xy_full_edge_kaggle_long24_full.yaml"),
        "node_indices": [0, 3, 4, 5, 6],
        "edge_indices": [0, 1, 2, 3, 4],
        "node_dim": 5,
        "edge_dim": 5,
    },
}

def load_resolved_config(config_path):
    from scripts.common import load_config
    return load_config(config_path, environment="kaggle")

def assert_config(tag, spec):
    assert spec["path"].exists(), f"Missing config: {spec['path']}"
    raw = spec["path"].read_text(encoding="utf-8")
    print(f"\n===== RAW {tag}: {spec['path']} =====")
    print(raw)
    cfg = load_resolved_config(spec["path"])
    print(f"===== RESOLVED {tag} =====")
    print(yaml.safe_dump(cfg, sort_keys=False)[:6000])
    assert cfg.get("experiment", {}).get("name") == spec["name"], cfg.get("experiment")
    fa = cfg.get("feature_ablation", {})
    assert fa.get("enabled") is True, fa
    assert [int(x) for x in fa.get("node_indices", [])] == spec["node_indices"], fa
    assert [int(x) for x in fa.get("edge_indices", [])] == spec["edge_indices"], fa
    assert int(cfg.get("model", {}).get("node_dim")) == spec["node_dim"], cfg.get("model")
    assert int(cfg.get("model", {}).get("edge_dim")) == spec["edge_dim"], cfg.get("model")
    assert cfg.get("teacher_alignment", {}).get("enabled") is False, cfg.get("teacher_alignment")
    assert int(cfg.get("training", {}).get("epochs")) == 24, cfg.get("training")
    assert cfg.get("training", {}).get("max_train_batches") in (None, "null"), cfg.get("training")
    assert cfg.get("training", {}).get("max_val_batches") in (None, "null"), cfg.get("training")
    assert "max_train_batches: 1" not in raw
    assert "max_val_batches: 1" not in raw
    print(f"Config {tag} OK")

for tag, spec in CONFIGS.items():
    assert_config(tag, spec)

In [ ]:
# Optional quick smoke, disabled by default.
# Khi bat RUN_SMOKE=True, smoke dung ten _smoke va output rieng, khong ghi de full run.
RUN_SMOKE = False

def run_cmd(cmd):
    print("\n$", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

if RUN_SMOKE:
    smoke_root = Path("/kaggle/working/outputs_d9_f0_be_long24_smoke")
    smoke_root.mkdir(parents=True, exist_ok=True)
    for tag, spec in CONFIGS.items():
        smoke_name = spec["name"] + "_smoke"
        smoke_dir = smoke_root / smoke_name
        cmd = [
            sys.executable, "-m", "scripts.train_motif_discovery_stage1",
            "--config", spec["path"],
            "--env", "kaggle",
            "--graph_repo_path", GRAPH_REPO_PATH,
            "--output_dir", smoke_dir,
            "--epochs", "1",
            "--experiment_name", smoke_name,
            "--device", "cuda",
            "--max_train_batches", "2",
            "--max_val_batches", "1",
            "--no_wandb",
        ]
        run_cmd(cmd)
else:
    print("RUN_SMOKE=False; skip optional capped smoke.")

In [ ]:
# Train B long24 full. Khong them max_train_batches/max_val_batches.
if OUTPUT_ROOT.exists() and not FORCE_OVERWRITE:
    RUN_OUTPUT_ROOT = OUTPUT_ROOT.with_name(OUTPUT_ROOT.name + "_" + time.strftime("%Y%m%d_%H%M%S"))
    print(f"OUTPUT_ROOT exists; using timestamped root: {RUN_OUTPUT_ROOT}")
else:
    RUN_OUTPUT_ROOT = OUTPUT_ROOT
RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RUN_DIRS = {tag: RUN_OUTPUT_ROOT / "runs" / spec["name"] for tag, spec in CONFIGS.items()}

B = CONFIGS["B"]
cmd_b = [
    sys.executable, "-m", "scripts.train_motif_discovery_stage1",
    "--config", B["path"],
    "--env", "kaggle",
    "--graph_repo_path", GRAPH_REPO_PATH,
    "--output_dir", RUN_DIRS["B"],
    "--epochs", "24",
    "--experiment_name", B["name"],
    "--device", "cuda",
    "--no_wandb",
]
run_cmd(cmd_b)

In [ ]:
# Train E long24 full. Khong them max_train_batches/max_val_batches.
E = CONFIGS["E"]
cmd_e = [
    sys.executable, "-m", "scripts.train_motif_discovery_stage1",
    "--config", E["path"],
    "--env", "kaggle",
    "--graph_repo_path", GRAPH_REPO_PATH,
    "--output_dir", RUN_DIRS["E"],
    "--epochs", "24",
    "--experiment_name", E["name"],
    "--device", "cuda",
    "--no_wandb",
]
run_cmd(cmd_e)

In [ ]:
import pandas as pd

def read_yaml(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

def flatten_metrics_from_history(history_path):
    df = pd.read_csv(history_path)
    if "split" in df.columns:
        train_rows = int((df["split"] == "train").sum())
        val_rows = int((df["split"] == "val").sum())
    else:
        train_rows = int(len(df))
        val_rows = 0
    epochs_run = int(df["epoch"].nunique()) if "epoch" in df.columns else int(len(df))
    best_epoch = None
    if "val_motif_quality_score" in df.columns and "epoch" in df.columns:
        val_df = df[df.get("split") == "val"] if "split" in df.columns else df
        s = pd.to_numeric(val_df["val_motif_quality_score"], errors="coerce")
        if s.notna().any():
            best_epoch = int(val_df.iloc[int(s.argmax())]["epoch"])
    return df, train_rows, val_rows, epochs_run, best_epoch

def validate_one_run(tag, spec):
    run_dir = RUN_DIRS[tag]
    history_path = run_dir / "logs" / "stage1_history.csv"
    resolved_path = run_dir / "resolved_config.yaml"
    ckpt_dir = run_dir / "checkpoints"
    print(f"\n===== Validate {tag}: {run_dir} =====")
    assert run_dir.exists(), f"Missing run dir: {run_dir}"
    assert history_path.exists(), f"Missing history: {history_path}"
    assert resolved_path.exists(), f"Missing resolved config: {resolved_path}"
    for ckpt_name in ("initial.pth", "best.pth", "last.pth"):
        assert (ckpt_dir / ckpt_name).exists(), f"Missing checkpoint: {ckpt_dir / ckpt_name}"
    raw_resolved = resolved_path.read_text(encoding="utf-8")
    cfg = read_yaml(resolved_path)
    train_cap = cfg.get("training", {}).get("max_train_batches")
    val_cap = cfg.get("training", {}).get("max_val_batches")
    if train_cap in (1, "1") or val_cap in (1, "1"):
        raise RuntimeError(f"Smoke batch cap leaked into full run: train={train_cap} val={val_cap}")
    if train_cap is not None or val_cap is not None:
        raise RuntimeError(f"Full run has batch cap: train={train_cap} val={val_cap}")
    assert cfg.get("teacher_alignment", {}).get("enabled") is False
    assert cfg.get("feature_ablation", {}).get("node_indices") == spec["node_indices"]
    assert cfg.get("feature_ablation", {}).get("edge_indices") == spec["edge_indices"]
    assert int(cfg.get("model", {}).get("node_dim")) == spec["node_dim"]
    assert int(cfg.get("model", {}).get("edge_dim")) == spec["edge_dim"]
    assert "max_train_batches: 1" not in raw_resolved
    assert "max_val_batches: 1" not in raw_resolved
    df, train_rows, val_rows, epochs_run, best_epoch = flatten_metrics_from_history(history_path)
    print({"train_rows": train_rows, "val_rows": val_rows, "epochs_run": epochs_run, "best_epoch": best_epoch})
    if epochs_run < 24 or train_rows < 24 or ("split" in df.columns and val_rows < 24):
        raise RuntimeError(f"History does not show full 24 epochs for {tag}")
    return {"run_dir": run_dir, "history_path": history_path, "resolved_path": resolved_path, "train_rows": train_rows, "val_rows": val_rows, "epochs_run": epochs_run, "best_epoch": best_epoch}

RUN_VALIDATION = {tag: validate_one_run(tag, spec) for tag, spec in CONFIGS.items()}

In [ ]:
# Visualize initial/best/last for B/E.
VIS_ROOT = RUN_OUTPUT_ROOT / "visualizations"
VIS_RESULTS = {}
for tag, spec in CONFIGS.items():
    VIS_RESULTS[tag] = {}
    for ckpt_type in ("initial", "best", "last"):
        ckpt_path = RUN_DIRS[tag] / "checkpoints" / f"{ckpt_type}.pth"
        out_dir = VIS_ROOT / tag / ckpt_type
        cmd = [
            sys.executable, "-m", "scripts.visualize_motif_discovery",
            "--config", spec["path"],
            "--env", "kaggle",
            "--graph_repo_path", GRAPH_REPO_PATH,
            "--checkpoint", ckpt_path,
            "--output_dir", out_dir,
            "--tag", ckpt_type,
            "--device", "cuda",
            "--split", "val",
            "--max_samples", "8",
            "--max_batches", "2",
        ]
        run_cmd(cmd)
        VIS_RESULTS[tag][ckpt_type] = out_dir

In [ ]:
# Separability audit for B best/last and E best/last.
# Optional Stage1G baseline: dien path neu muon audit baseline trong cung notebook.
STAGE1G_CHECKPOINT = None
STAGE1G_CONFIG = None

AUDIT_ROOT = RUN_OUTPUT_ROOT / "separability"
TMP_AUDIT_CONFIG_ROOT = Path("/kaggle/working/tmp_d9_f0_audit_configs")
TMP_AUDIT_CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
AUDIT_RESULTS = {}

def make_audit_config(name, stage1_config, stage1_checkpoint, out_dir):
    cfg = load_resolved_config(stage1_config)
    audit_cfg = {
        "experiment": {"name": name},
        "paths": {
            "graph_repo_path": str(GRAPH_REPO_PATH),
            "resolved_output_root": str(out_dir),
        },
        "output": {"dir": str(out_dir)},
        "data": cfg.get("data", {}),
        "training": {"device": "cuda", "seed": cfg.get("training", {}).get("seed", 42)},
        "stage1": {"config": str(stage1_config), "checkpoint": str(stage1_checkpoint)},
        "stage2": {
            "selection_source": "selected_weights",
            "top_m": int(cfg.get("stage2", {}).get("top_m", cfg.get("model", {}).get("top_m", 8))),
        },
        "environments": {
            "kaggle": {
                "paths": {
                    "graph_repo_path": str(GRAPH_REPO_PATH),
                    "resolved_output_root": str(out_dir),
                }
            }
        },
    }
    cfg_path = TMP_AUDIT_CONFIG_ROOT / f"{name}.yaml"
    cfg_path.write_text(yaml.safe_dump(audit_cfg, sort_keys=False), encoding="utf-8")
    return cfg_path

def run_audit(label, stage1_config, stage1_checkpoint):
    out_dir = AUDIT_ROOT / label
    cfg_path = make_audit_config(label, stage1_config, stage1_checkpoint, out_dir)
    cmd = [
        sys.executable, "-m", "scripts.audit_frozen_motif_separability",
        "--config", cfg_path,
        "--env", "kaggle",
        "--graph_repo_path", GRAPH_REPO_PATH,
        "--output_dir", out_dir,
        "--device", "cuda",
        "--train_batches", "60",
        "--val_batches", "60",
    ]
    try:
        run_cmd(cmd)
        return {"config": cfg_path, "out_dir": out_dir, "metrics": out_dir / "metrics.json", "summary": out_dir / "summary.csv", "status": "ok"}
    except Exception as exc:
        print(f"[AUDIT WARNING] {label} failed: {exc}")
        return {"config": cfg_path, "out_dir": out_dir, "metrics": None, "summary": None, "status": f"failed: {exc}"}

for tag, spec in CONFIGS.items():
    for ckpt_type in ("best", "last"):
        label = f"{tag}_{ckpt_type}"
        AUDIT_RESULTS[label] = run_audit(label, spec["path"], RUN_DIRS[tag] / "checkpoints" / f"{ckpt_type}.pth")

if STAGE1G_CHECKPOINT and STAGE1G_CONFIG:
    AUDIT_RESULTS["Stage1G_best"] = run_audit("Stage1G_best", Path(STAGE1G_CONFIG), Path(STAGE1G_CHECKPOINT))
else:
    print("Stage1G baseline audit skipped: STAGE1G_CHECKPOINT/STAGE1G_CONFIG not set.")

In [ ]:
SUMMARY_PATH = RUN_OUTPUT_ROOT / "d9_f0_be_kaggle_long24_full_summary.csv"
SUMMARY_COLUMNS = [
    "run_name", "checkpoint_type", "config_path", "checkpoint_path", "history_path", "visualization_path", "separability_path",
    "node_indices", "edge_indices", "node_dim", "edge_dim", "epochs_run", "train_rows", "val_rows", "best_epoch",
    "selected_border", "selected_outer", "selected_foreground", "map_sim", "redundant_ratio", "selection_entropy", "selection_effective_count",
    "clean", "clean_count", "upper_clean", "middle_clean", "lower_clean", "coverage_cosine",
    "knn1", "knn5", "centroid_cosine", "intra_cosine", "inter_cosine", "intra_inter_gap", "notes",
]

def latest_val_metric(history_path, column):
    try:
        df = pd.read_csv(history_path)
        if "split" in df.columns:
            df = df[df["split"] == "val"]
        if column not in df.columns or df.empty:
            return pd.NA
        s = pd.to_numeric(df[column], errors="coerce").dropna()
        return s.iloc[-1] if len(s) else pd.NA
    except Exception:
        return pd.NA

def load_audit_metrics(label):
    item = AUDIT_RESULTS.get(label, {})
    path = item.get("metrics")
    if path and Path(path).exists():
        return json.loads(Path(path).read_text(encoding="utf-8"))
    return {}

rows = []
for tag, spec in CONFIGS.items():
    valid = RUN_VALIDATION[tag]
    for ckpt_type in ("initial", "best", "last"):
        label = f"{tag}_{ckpt_type}"
        metrics = load_audit_metrics(label)
        train_cos = metrics.get("train_cosine", {})
        val_cos = metrics.get("val_cosine", {})
        intra = val_cos.get("intra_class_cosine_mean", pd.NA)
        inter = val_cos.get("inter_class_cosine_mean", pd.NA)
        try:
            gap = float(intra) - float(inter)
        except Exception:
            gap = pd.NA
        rows.append({
            "run_name": spec["name"],
            "checkpoint_type": ckpt_type,
            "config_path": str(spec["path"]),
            "checkpoint_path": str(RUN_DIRS[tag] / "checkpoints" / f"{ckpt_type}.pth"),
            "history_path": str(valid["history_path"]),
            "visualization_path": str(VIS_RESULTS.get(tag, {}).get(ckpt_type, pd.NA)),
            "separability_path": str(AUDIT_RESULTS.get(label, {}).get("out_dir", pd.NA)) if ckpt_type in ("best", "last") else pd.NA,
            "node_indices": json.dumps(spec["node_indices"]),
            "edge_indices": json.dumps(spec["edge_indices"]),
            "node_dim": spec["node_dim"],
            "edge_dim": spec["edge_dim"],
            "epochs_run": valid["epochs_run"],
            "train_rows": valid["train_rows"],
            "val_rows": valid["val_rows"],
            "best_epoch": valid["best_epoch"],
            "selected_border": latest_val_metric(valid["history_path"], "selected_border_mass_mean"),
            "selected_outer": latest_val_metric(valid["history_path"], "selected_outer_border_mass_mean"),
            "selected_foreground": latest_val_metric(valid["history_path"], "selected_foreground_mass_mean"),
            "map_sim": latest_val_metric(valid["history_path"], "mean_pairwise_map_sim"),
            "redundant_ratio": latest_val_metric(valid["history_path"], "redundant_pair_ratio"),
            "selection_entropy": latest_val_metric(valid["history_path"], "selection_entropy"),
            "selection_effective_count": latest_val_metric(valid["history_path"], "selection_effective_count"),
            "clean": latest_val_metric(valid["history_path"], "clean_score_mean"),
            "clean_count": latest_val_metric(valid["history_path"], "clean_candidate_count"),
            "upper_clean": latest_val_metric(valid["history_path"], "upper_clean_count"),
            "middle_clean": latest_val_metric(valid["history_path"], "middle_clean_count"),
            "lower_clean": latest_val_metric(valid["history_path"], "lower_clean_count"),
            "coverage_cosine": latest_val_metric(valid["history_path"], "coverage_cosine"),
            "knn1": metrics.get("knn1_accuracy", pd.NA),
            "knn5": metrics.get("knn5_accuracy", pd.NA),
            "centroid_cosine": metrics.get("mean_centroid_cosine", pd.NA),
            "intra_cosine": intra,
            "inter_cosine": inter,
            "intra_inter_gap": gap,
            "notes": AUDIT_RESULTS.get(label, {}).get("status", "audit_not_run_for_initial"),
        })
summary_df = pd.DataFrame(rows, columns=SUMMARY_COLUMNS)
SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(SUMMARY_PATH, index=False)
print("Summary CSV:", SUMMARY_PATH)
display(summary_df)

In [ ]:
ANALYSIS_PATH = RUN_OUTPUT_ROOT / "d9_f0_be_kaggle_long24_full_analysis.md"

def na_to_text(v):
    if pd.isna(v):
        return "NA"
    return str(v)

summary_records = pd.read_csv(SUMMARY_PATH).to_dict("records")
lines = []
lines.append("# D9-F0 B/E Kaggle Long24 Full Analysis\n")
lines.append("## 1. Run validity\n")
lines.append(f"- GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA unavailable'}")
lines.append(f"- graph_repo path: {GRAPH_REPO_PATH}")
for tag, spec in CONFIGS.items():
    valid = RUN_VALIDATION[tag]
    lines.append(f"- {tag}: feature mask node={spec['node_indices']} edge={spec['edge_indices']}; teacher_alignment=false; epochs_run={valid['epochs_run']}; train_rows={valid['train_rows']}; val_rows={valid['val_rows']}; checkpoints initial/best/last exist")
lines.append("- Batch cap: none in full run resolved configs")
lines.append("")

for section, tag in (("## 2. B result", "B"), ("## 3. E result", "E")):
    lines.append(section)
    for rec in summary_records:
        if rec["run_name"] == CONFIGS[tag]["name"] and rec["checkpoint_type"] in ("best", "last"):
            lines.append(f"- {rec['checkpoint_type']}: selected_border={na_to_text(rec['selected_border'])}, selected_outer={na_to_text(rec['selected_outer'])}, selected_foreground={na_to_text(rec['selected_foreground'])}, clean_count={na_to_text(rec['clean_count'])}, knn1={na_to_text(rec['knn1'])}, knn5={na_to_text(rec['knn5'])}, notes={rec['notes']}")
    lines.append(f"- Visualization path: {VIS_ROOT / tag}")
    lines.append("")

lines.append("## 4. B vs E")
lines.append("- Compare B/E only from the summary CSV metrics produced by this notebook.")
lines.append("- Check whether B has stronger border/outer/foreground behavior and whether that looks like spatial shortcut in visualizations.")
lines.append("- Check whether E improves clean/clean_count/separability and whether E drifts toward border-heavy motifs.")
lines.append("")
lines.append("## 5. So voi Stage1G baseline")
if "Stage1G_best" in AUDIT_RESULTS:
    lines.append(f"- Stage1G audit path: {AUDIT_RESULTS['Stage1G_best'].get('out_dir')}")
else:
    lines.append("- Chua so sanh truc tiep duoc trong notebook nay vi STAGE1G_CHECKPOINT/STAGE1G_CONFIG chua duoc set.")
lines.append("")
lines.append("## 6. Recommendation")
lines.append("- Ket luan chi nen dua tren metrics/visualization/audit da sinh trong run nay.")
lines.append("- Lua chon hop le: chon B, chon E, giu ca B/E cho D9-RG mini, hoac dung feature ablation de chuyen sang D9-RG/Stability.")
lines.append("- Khong ket luan feature nao tot nhat tuyet doi tu notebook nay.")
ANALYSIS_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Analysis Markdown:", ANALYSIS_PATH)
print(ANALYSIS_PATH.read_text(encoding="utf-8"))

In [ ]:
ZIP_PATH = Path("/kaggle/working/d9_f0_be_kaggle_long24_full_outputs.zip")
if ZIP_PATH.exists() and not FORCE_OVERWRITE:
    ZIP_PATH = ZIP_PATH.with_name(ZIP_PATH.stem + "_" + time.strftime("%Y%m%d_%H%M%S") + ZIP_PATH.suffix)
archive_base = ZIP_PATH.with_suffix("")
shutil.make_archive(str(archive_base), "zip", root_dir=RUN_OUTPUT_ROOT)
print("Zip output:", ZIP_PATH)